# Make a quantgap pack on a free Google Colab GPU

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saqlain2204/quantgap/blob/main/notebooks/make_pack_colab.ipynb)

A **pack** holds the original model's answers on a fixed set of texts. You make it once, then everyone can score quants of that model without running the big model.

Before you run:
1. Menu **Runtime > Change runtime type > T4 GPU** (free tier).
2. To upload, add a Colab secret named `HF_TOKEN` (key icon in the left bar, turn on *Notebook access*). It needs write access to the place you upload to.
3. Check the model's license allows sharing its outputs. Do not publish a pack for a model whose license forbids it.

The free T4 has 16 GB of GPU memory and about 12 GB of CPU memory. Models up to about 3B parameters fit comfortably. A 7B model in 16-bit is very tight. Bigger models need a larger machine. The T4 has no native bfloat16, so it is slower than newer GPUs.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # change me
REVISION = ""  # full commit sha from the model page (recommended)
LICENSE = "apache-2.0"  # the model's license
PUSH_TO = ""  # e.g. "quantgap-packs/Qwen2.5-0.5B-Instruct"; empty = do not upload

In [ ]:
!pip install -q "quantgap[pack]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import subprocess

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret found: upload will not work.")

cmd = ["quantgap", "pack", MODEL, "--license", LICENSE, "--out", "/content/pack"]
if REVISION:
    cmd += ["--revision", REVISION]
if PUSH_TO:
    cmd += ["--push", PUSH_TO]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
!quantgap packs show /content/pack
!du -sh /content/pack

Download `/content/pack` from the file browser on the left (or zip it first with `!zip -r pack.zip /content/pack`), or use the uploaded dataset with:

```
quantgap score <your quant> --pack <org>/<dataset>
```

Colab sessions end after a while, so download or upload the pack before closing the tab.